# Phase 1 — Baseline Inference (Qwen3-4B-Thinking-2507, bf16, vLLM)

Single-sample baseline. The honest number to beat before self-consistency (Phase 2) or training (Phase 3+).

**Design choices** (see planning discussion):
- **bf16**, not INT4/INT8 — no benefit on A100 80GB, quantization hurts math reasoning.
- Sampling = model-card thinking defaults: `temp 0.6, top_p 0.95, top_k 20, min_p 0`, **N=1**.
- Raw output kept (no reasoning parser) — submission + grader need the full trace.
- `MAX_GEN_TOKENS = 32768` (model-card "general"); **truncation rate is logged** so we bump only if data justifies it.
- **Precision-aware prompts**: exact fractions/symbolic forms, ≥10 sig figs otherwise, clean LaTeX, single trailing `\boxed{}`.

**Prereqs in the Colab session:** upload `judger.py`, `utils.py`, `harness.py`, `public.jsonl`, `val_ids.json`, `train_ids.json` to the working directory (or mount from Drive).

## 0. Mount Drive

Project files (`judger.py`, `utils.py`, `harness.py`, `public.jsonl`, `val_ids.json`, `train_ids.json`) live in the Drive folder below. Adjust the path to your folder.

In [1]:
import sys
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
PROJECT_DIR = '/content/drive/MyDrive/second_try/'
sys.path.append(PROJECT_DIR)

In [3]:
import utils
import judger
import harness as H

## 1. Environment — coherent version set

The crashes were a **three-way mismatch** between `torch`, `transformers`, and `vllm`; fixing one moved the break to another. The cure is to pin all three as a coherent set. There are two such sets:

**Option A (default — native Qwen3 support, faster):**
- `vllm==0.9.2` — has native `Qwen3ForCausalLM` kernels (the official floor for Qwen3-2507 is vllm≥0.8.5)
- `torch==2.7.0` — exactly what vllm 0.9.2 pins
- `transformers==4.53.3` — has Qwen3 (added 4.51.0), and is **below 4.54** (transformers ≥4.54 adds a native `aimv2` config that collides with vllm 0.9.2's own registration → `ValueError: 'aimv2' is already used`). The window `[4.51.1, 4.54)` is the sweet spot.

**Option B (fallback — the recipe that already worked for you):** torch 2.5.1 + vllm 0.7.3, which runs Qwen3 via vLLM's generic *Transformers backend* (works, but slower — no Qwen3-specific kernels).

Run **Option A first**. If its install fails (e.g. torch 2.7 wheel vs. Colab's CUDA driver — try switching the index from `cu126` to `cu128`), comment A and run B. Either way: **after the install cell finishes the first time, Runtime → Restart, then run from section 1b** (packages persist; the restart makes the pinned torch/transformers the ones actually imported).

To confirm Option A got the *native* Qwen3 path (not the slow Transformers-backend fallback), check the model-load log in section 4 for `Resolved architecture: Qwen3ForCausalLM` with no "Transformers backend" / "TransformersForCausalLM" note.

**Note:** plain vLLM for inference only — do NOT install Unsloth here (training tool, Phases 3/4; conflicts with standalone vLLM).

In [4]:
# ===== OPTION A: native Qwen3 support (default) =====
import os

# transformers must be >=4.51.1 (Qwen3 support) AND <4.54 (>=4.54 adds a native
# "aimv2" config that collides with vllm 0.9.2's own aimv2 registration ->
# "ValueError: 'aimv2' is already used by a Transformers config").
constraints = """torch==2.7.0
transformers==4.53.3
"""
with open("/content/constraints.txt", "w") as f:
    f.write(constraints)

!wget -qO- https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]

# torch 2.7.0 (cu126 wheels — broad driver compatibility; switch to cu128 if needed).
!uv pip install --system torch==2.7.0 --index-url https://download.pytorch.org/whl/cu126
!uv pip install --system \
    "vllm==0.9.2" "transformers==4.53.3" \
    sympy numpy tqdm antlr4-python3-runtime==4.11.1 accelerate \
    -c /content/constraints.txt

print("Option A install done. RESTART THE RUNTIME, then run from section 1b.")

downloading uv 0.11.16 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
Using Python 3.12.13 environment at: /usr
Resolved 25 packages in 1.37s
Prepared 16 packages in 1m 02s
Uninstalled 16 packages in 705ms
Installed 16 packages in 199ms
 - nvidia-cublas-cu12==12.8.4.1
 + nvidia-cublas-cu12==12.6.4.1
 - nvidia-cuda-cupti-cu12==12.8.90
 + nvidia-cuda-cupti-cu12==12.6.80
 - nvidia-cuda-nvrtc-cu12==12.8.93
 + nvidia-cuda-nvrtc-cu12==12.6.77
 - nvidia-cuda-runtime-cu12==12.8.90
 + nvidia-cuda-runtime-cu12==12.6.77
 - nvidia-cudnn-cu12==9.10.2.21
 + nvidia-cudnn-cu12==9.5.1.17
 - nvidia-cufft-cu12==11.3.3.83
 + nvidia-cufft-cu12==11.3.0.4
 - nvidia-cufile-cu12==1.13.1.3
 + nvidia-cufile-cu12==1.11.1.6
 - nvidia-curand-cu12==10.3.9.90
 + nvidia-curand-cu12==10.3.7.77
 - nvidia-cusolver-cu12==11.7.3.90
 + nvidia-cusolver-cu12==11.7.1.2
 - nvidia-cusparse-cu12==12.5.8.93
 + nvidia-cusparse-cu12==12.5.4.2
 - nvidia-cusparselt-cu12==0.7.1
 + nvidia-cuspar

### Option B — fallback (only if A fails to install)

Your known-good recipe: torch 2.5.1 forces vllm back to 0.7.3, which runs Qwen3 via the generic Transformers backend. Slower but proven. Run this cell *instead of* the Option A cell, then restart.

In [5]:
# # ===== OPTION B: fallback (run only if Option A install failed) =====
# import os

# constraints = """torch==2.5.1
# torchvision==0.20.1
# torchaudio==2.5.1
# transformers>=4.48,<=4.57
# """
# with open("/content/constraints.txt", "w") as f:
#     f.write(constraints)

# !wget -qO- https://astral.sh/uv/install.sh | sh
# os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]

# !uv pip install --system torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 \
#     --index-url https://download.pytorch.org/whl/cu121
# !uv pip install --system \
#     sympy numpy transformers vllm tqdm \
#     antlr4-python3-runtime==4.11.1 accelerate \
#     -c /content/constraints.txt

# print("Option B install done. RESTART THE RUNTIME, then run from section 1b.")

## 1b. Post-restart checks + engine flags (run BEFORE anything touches CUDA)

`spawn` avoids the fork-vs-CUDA conflict. Importing torch/vllm here to print versions is fine — but note that touching `torch.cuda` does init CUDA, so this is the *first* CUDA-touching cell by design, and the model load follows directly.

In [6]:
import os
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
# If the engine still fails at load, uncomment to run in-process so the REAL
# exception prints here instead of a swallowed subprocess error, then re-run sec 4:
# os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

import torch, transformers, vllm
print("torch:", torch.__version__, "| transformers:", transformers.__version__, "| vllm:", vllm.__version__)
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0))
print("GPU free:", round(torch.cuda.mem_get_info(0)[0] / 1e9, 2), "GB")
assert transformers.__version__.split(".")[0] == "4", "transformers must be 4.x — restart runtime"

torch: 2.7.0+cu126 | transformers: 4.53.3 | vllm: 0.9.2
CUDA: True | Tesla T4
GPU free: 15.53 GB


## 2. Config

In [7]:
import json, os, sys

MODEL_ID       = "Qwen/Qwen3-4B-Thinking-2507"
DATA_PATH      = os.path.join(PROJECT_DIR, "public.jsonl")
VAL_IDS_PATH   = os.path.join(PROJECT_DIR, "val_ids.json")
TRAIN_IDS_PATH = os.path.join(PROJECT_DIR, "train_ids.json")

MAX_GEN_TOKENS = 32768     # <- single knob. Bump (49152 / 81920) only if truncation is high.
MAX_MODEL_LEN  = 40960     # must exceed (longest prompt + MAX_GEN_TOKENS). Raise with MAX_GEN_TOKENS.
GPU_MEM_UTIL   = 0.85      # conservative; raise toward 0.92 only if you need KV headroom
SEED           = 151

# Write outputs back to Drive so they survive runtime resets.
PRED_JSONL     = os.path.join(PROJECT_DIR, "phase1_predictions.jsonl")
SUBMISSION_CSV = os.path.join(PROJECT_DIR, "phase1_submission.csv")

data = [json.loads(l) for l in open(DATA_PATH)]
print(f"Loaded {len(data)} rows")

Loaded 1126 rows


## 3. Prompts & helpers

Tightened vs the starter to enforce the grader's precision + format rules (verified against `judger.py`).

In [8]:
from typing import Optional

SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Give your final answer inside a single \\boxed{}. "
    "Use EXACT values: prefer fractions (\\frac{a}{b}) and symbolic forms "
    "(\\sqrt{}, \\pi, e) over decimals. If you must give a decimal, write at "
    "least 10 significant figures and do NOT round. "
    "If the problem has multiple sub-answers, put them all inside one \\boxed{}, "
    "comma-separated, in the order asked, e.g. \\boxed{41, 35, 16}. "
    "If a single sub-answer itself contains a comma (a point or tuple), wrap it "
    "in parentheses, e.g. \\boxed{(2, 3), 7}."
)
SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. Read the problem and the answer choices, "
    "then select the single best answer. After your reasoning, output ONLY the "
    "letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}. "
    "The very last thing in your response must be that \\boxed{<letter>}."
)

def build_prompt(question: str, options: Optional[list]):
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question

def build_chat_messages(question, options):
    s, u = build_prompt(question, options)
    return [{"role": "system", "content": s}, {"role": "user", "content": u}]

## 4. Load model (bf16) & build prompts

`enforce_eager=True` skips CUDA-graph capture (a common Colab init-failure point; the graph speedup is marginal for a one-shot batch job).

**If you still get "Engine core initialization failed":** the real error is in the swallowed subprocess. Go to section 1b, uncomment `VLLM_ENABLE_V1_MULTIPROCESSING="0"`, restart the runtime, and re-run — the true exception prints here. Common real causes: a torch/transformers/vllm version mismatch (the pinned set in section 1 + the restart is the fix), OOM (lower `GPU_MEM_UTIL` or `MAX_MODEL_LEN`).

In [12]:
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

prompts, order = [], []
for r in data:
    msgs = build_chat_messages(r["question"], r.get("options"))
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    prompts.append(text); order.append(r["id"])

# Quick prompt-length sanity: ensure MAX_MODEL_LEN headroom is real.
plens = [len(tok(p).input_ids) for p in prompts]
print(f"prompt tokens: max={max(plens)} p95={sorted(plens)[int(0.95*len(plens))]} "
      f"| headroom for gen = {MAX_MODEL_LEN - max(plens)} (need >= {MAX_GEN_TOKENS})")
assert MAX_MODEL_LEN - max(plens) >= MAX_GEN_TOKENS, "Raise MAX_MODEL_LEN: prompt+gen exceeds context"

# llm = LLM(model=MODEL_ID, dtype="bfloat16", trust_remote_code=True,
#           max_model_len=MAX_MODEL_LEN, gpu_memory_utilization=GPU_MEM_UTIL,
#           seed=SEED, enforce_eager=True)

INFO 05-26 06:48:49 [__init__.py:244] Automatically detected platform cuda.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

prompt tokens: max=4156 p95=759 | headroom for gen = 36804 (need >= 32768)


## 4b. Smoke test — one row before committing to all 1126

vllm 0.7.3 (forced by the torch-2.5.1 pin) predates the official Qwen3-2507 support
floor, so we verify on a single prompt that generation produces a non-empty trace with
a `</think>` and a `\boxed{}` before spending compute on the full set. If this fails or
looks wrong, stop — do not run section 5.

In [10]:
# _sp = SamplingParams(temperature=0.6, top_p=0.95, top_k=20, min_p=0.0, max_tokens=2048)
# _out = llm.generate([prompts[0]], _sp)[0].outputs[0]
# _txt = _out.text
# print("finish_reason:", _out.finish_reason, "| gen tokens:", len(_out.token_ids))
# print("has </think>:", "</think>" in _txt, "| has \\boxed:", "\\boxed" in _txt)
# print("--- first 300 chars ---\n", _txt[:300])
# print("--- last 300 chars ---\n", _txt[-300:])
# assert len(_txt.strip()) > 0, "EMPTY generation — vllm/model mismatch; do not proceed"

## 5. Generate (single sample)

In [ ]:
# sp = SamplingParams(temperature=0.6, top_p=0.95, top_k=20, min_p=0.0, max_tokens=MAX_GEN_TOKENS)
# outs = llm.generate(prompts, sp)

In [18]:
import json

In [36]:
# results = []
# for rid, out in zip(order, outs):
#     o = out.outputs[0]
#     results.append({
#         "id": rid,
#         "response": o.text,
#         "finish_reason": o.finish_reason,
#         "n_gen_tokens": len(o.token_ids),
#         "truncated": (o.finish_reason == "length"),
#     })
with open(PROJECT_DIR + 'phase_1_preds/phase1_predictions.jsonl') as f:
    results = [json.loads(line) for line in f if line.strip()]

n_trunc = sum(r["truncated"] for r in results)
print(f"Generated {len(results)} | truncated (hit token cap): {n_trunc} "
      f"({100*n_trunc/len(results):.1f}%)")
print("If truncation is high (>~4%) or concentrated in free_multi, raise MAX_GEN_TOKENS.")

Generated 1126 | truncated (hit token cap): 11 (1.0%)
If truncation is high (>~4%) or concentrated in free_multi, raise MAX_GEN_TOKENS.


## 6. Save predictions + submission CSV

In [ ]:
# import csv

# with open(PRED_JSONL, "w") as f:
#     for r in results:
#         f.write(json.dumps(r) + "\n")

# with open(SUBMISSION_CSV, "w", newline="") as f:
#     w = csv.writer(f, quoting=csv.QUOTE_ALL)
#     w.writerow(["id", "response"])
#     for r in sorted(results, key=lambda x: int(x["id"])):
#         w.writerow([r["id"], r["response"]])

# print(f"Wrote {PRED_JSONL} and {SUBMISSION_CSV}")

## 7. Score on held-out validation (the number that matters)

Uses the Phase-0 harness (`harness.py`), which wraps the official grader exactly.
Headline = VAL (held-out 169). TRAIN buckets shown separately for diagnostics — **do not tune on them.**

In [37]:
import time
import signal

# -----------------------------
# Build lookup maps (once)
# -----------------------------
data_by_id = {int(r["id"]): r for r in data}
res_map = {int(r["id"]): r for r in results}
trunc_ids = {int(r["id"]) for r in results if r["truncated"]}

val_ids = set(json.load(open(VAL_IDS_PATH)))
train_ids = set(json.load(open(TRAIN_IDS_PATH)))


# -----------------------------
# Timeout wrapper (prevents hangs)
# -----------------------------
class Timeout(Exception):
    pass

def _timeout_handler(signum, frame):
    raise Timeout()

signal.signal(signal.SIGALRM, _timeout_handler)

def safe_score_one(row, resp, judger, pid, timeout=2):
    signal.alarm(timeout)
    try:
        t0 = time.time()
        out = H.score_one(row, resp, judger)
        signal.alarm(0)
        return out, time.time() - t0
    except Timeout:
        signal.alarm(0)
        print(f"  TIMEOUT id={pid} (likely SymPy / norm_math_str)")
        return None, timeout
    except Exception as e:
        signal.alarm(0)
        print(f"  ERROR id={pid}: {e}")
        return None, 0.0


# -----------------------------
# Main scorer
# -----------------------------
def score_with_progress(name, id_set):
    print(f"\n{'='*60}")
    print(f"SCORING: {name}")
    print(f"{'='*60}")

    judger = H.Judger(strict_extract=False)

    ids = [
        int(pid)
        for pid in id_set
        if int(pid) in data_by_id and int(pid) in res_map
    ]

    per_row = []
    bucket_counts = {"mc": 0, "free_single": 0, "free_multi": 0}

    t_start = time.time()

    for i, pid in enumerate(ids, 1):

        row = data_by_id[pid]
        resp = res_map[pid]["response"]

        # progress print
        if i % 10 == 0 or i == len(ids):
            elapsed = time.time() - t_start
            rate = i / elapsed if elapsed > 0 else 0
            print(f"[{i:4d}/{len(ids)}] id={pid} "
                  f"elapsed={elapsed:.1f}s rate={rate:.2f} rows/s")

        result, dt = safe_score_one(row, resp, judger, pid, timeout=2)

        if result is None:
            continue

        result["truncated"] = (pid in trunc_ids)

        per_row.append(result)

        # bucket tracking (live)
        bucket_counts[result["bucket"]] += 1

        if dt > 1.5:
            print(f"  SLOW ⚠️ id={pid} took {dt:.2f}s (bucket={result['bucket']})")

    # -----------------------------
    # Summary
    # -----------------------------
    summary = H.summarize(per_row)

    print("\n" + "="*60)
    print("FINAL SUMMARY")
    print("="*60)
    H.print_summary(summary)

    print("\nBucket counts (sanity check):")
    print(bucket_counts)

    return summary, per_row


# -----------------------------
# RUN
# -----------------------------
print("### VALIDATION ###")
val_summary, val_per = score_with_progress("VALIDATION", val_ids)

print("\n### TRAIN ###")
train_summary, train_per = score_with_progress("TRAIN", train_ids)

### VALIDATION ###

SCORING: VALIDATION
[  10/169] id=532 elapsed=0.8s rate=12.34 rows/s
[  20/169] id=40 elapsed=1.3s rate=15.19 rows/s
[  30/169] id=1083 elapsed=2.0s rate=15.25 rows/s
[  40/169] id=76 elapsed=2.3s rate=17.49 rows/s
[  50/169] id=1118 elapsed=2.5s rate=19.67 rows/s
[  60/169] id=125 elapsed=2.9s rate=20.91 rows/s
[  70/169] id=663 elapsed=3.5s rate=19.76 rows/s
[  80/169] id=178 elapsed=4.6s rate=17.53 rows/s
[  90/169] id=747 elapsed=5.3s rate=17.11 rows/s
[ 100/169] id=257 elapsed=5.7s rate=17.47 rows/s
[ 110/169] id=289 elapsed=6.1s rate=17.93 rows/s
[ 120/169] id=832 elapsed=6.7s rate=17.80 rows/s
[ 130/169] id=361 elapsed=7.2s rate=18.05 rows/s
[ 140/169] id=898 elapsed=8.5s rate=16.51 rows/s
[ 150/169] id=933 elapsed=9.3s rate=16.12 rows/s
[ 160/169] id=984 elapsed=10.7s rate=14.96 rows/s
[ 169/169] id=506 elapsed=11.1s rate=15.25 rows/s

FINAL SUMMARY
EVALUATION RESULTS
  MC         :   45 /   56  ( 80.36%)
  Free-1blank:   34 /   51  ( 66.67%)
  Free-multi : 

## 8. Inspect failures (where to spend Phase 2)

Look at MC-fallback firings (fragile letter extraction) and free-form misses by bucket.

In [38]:
mc_fallback = [d for d in per_val if d["is_mc"] and d["mc_fallback"]]
print(f"MC scored via fallback regex on VAL: {len(mc_fallback)} "
      f"(of which correct-but-fragile: {sum(d['correct'] for d in mc_fallback)})")
print("These are the rows where a cleaner boxed-letter prompt would de-risk scoring.\n")

misses = [d for d in per_val if not d["correct"]]
by_bucket = {}
for d in misses:
    by_bucket.setdefault(d["bucket"], []).append(d["id"])
print("VAL misses by bucket:")
for b, ids in sorted(by_bucket.items()):
    print(f"  {b}: {len(ids)}  ids={ids}")

MC scored via fallback regex on VAL: 1 (of which correct-but-fragile: 0)
These are the rows where a cleaner boxed-letter prompt would de-risk scoring.

VAL misses by bucket:
  free_multi: 20  ids=[12, 22, 1088, 584, 1103, 127, 673, 682, 684, 695, 703, 195, 775, 824, 843, 365, 373, 374, 414, 487]
  free_single: 17  ids=[523, 539, 1118, 622, 125, 644, 649, 159, 252, 763, 289, 805, 827, 841, 435, 976, 468]
  mc: 11  ids=[1045, 593, 138, 704, 257, 790, 279, 318, 830, 959, 474]
